# Images

#### Telecopes and detectors produce images 

#### We have arrays of pixel values, not lists of celestial objects 

#### How do we identify objects in the catalogues?

#### We look for groups of pixels that differ significantly from the background

#### We look for groups of pixels that have the right shape (i.e. blurred by the optics and atmosphere)

#### We want consistent identification of objects - automated rather than by eye

#### 

# Segmentation 

#### Identify pixels that meet a signal to noise threshold

#### Connect pixels together and create objects when the number of pixels is above some threshold

#### Can identify objects with somewhat arbitrary shapes 

#### Can struggle with diffuse objects where individual pixels may not be above the noise 

####

In [ ]:
# Import various libraries 

import numpy as np
import astropy
import photutils
import ccdproc
from ccdproc import CCDData, combiner
from astropy import units as u
import astropy.io.fits as fits
from astropy.io import ascii
from astropy.visualization import SqrtStretch
from astropy.visualization.mpl_normalize import ImageNormalize
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from photutils.centroids import centroid_com, centroid_1dg, centroid_2dg
from photutils.aperture import CircularAperture
from photutils.aperture import aperture_photometry
from photutils.background import Background2D
from photutils.background import MedianBackground
from photutils.detection import DAOStarFinder
from photutils.segmentation  import detect_sources, deblend_sources, SourceCatalog
from scipy.ndimage import shift
import gc                               

from astropy.coordinates import SkyCoord
from astroquery.gaia import Gaia


In [ ]:
fn='NGC_3293_V_median_wcs.fits'
science_im=CCDData.read(fn, unit = "adu")                           

skydata=science_im.data
print(skydata[0])
skydata=science_im.data[1030:1130,60:160]
p1=np.nanpercentile(skydata, 1)
p99=np.nanpercentile(skydata, 99)
plt.imshow(skydata, cmap='gray', vmin=p1, vmax=p99)
ax = plt.gca()
ax.invert_yaxis()
plt.colorbar(shrink=0.8)



In [ ]:
# Lets run stats on some of this data to get the background and noise

med=np.nanmedian(skydata)
print('Median:', med)

std=np.nanstd(skydata)
print('Standard deviation:', std)


In [ ]:
# Lets display the image with 2-simga corresponding to white 

#skydata=science_im.data
skydata=science_im.data[500:1000,500:1000]
p1=np.nanpercentile(skydata, 1)
p99=np.nanpercentile(skydata, 99)
plt.imshow(skydata, cmap='gray', vmin=1.99*std, vmax=2*std)
ax = plt.gca()
ax.invert_yaxis()
plt.colorbar(shrink=0.8)
plt.title('Black and white')
plt.show()

p1=np.nanpercentile(skydata, 1)
p99=np.nanpercentile(skydata, 99)
plt.imshow(skydata, cmap='gray', vmin=-2.0*std, vmax=2.0*std)
ax = plt.gca()
ax.invert_yaxis()
plt.colorbar(shrink=0.8)
plt.title('Greyscale')
plt.show()



# Segmentation image 

In [ ]:
segimage=detect_sources(science_im.data, 2.00*std, 9, connectivity=4, mask=None)
print('Segmentation image minimum:', np.min(segimage.data))
print('Segmentation image maximum:', np.max(segimage.data))


In [ ]:
skydata=segimage[500:1000,500:1000]
plt.imshow(skydata, cmap='gray')
ax = plt.gca()
ax.invert_yaxis()
plt.colorbar(shrink=0.8)
plt.title('Segmentation image')
plt.show()


In [ ]:
outfile="segimage.fits"                # Set the output file name
hdu=fits.PrimaryHDU(segimage)          # Define a FITS header for this data
hdu.writeto(outfile, overwrite=True)   # Write the output and overwrite existing files if needed. 


# What do the pixel values in the segmentation image correspond to?

# Read the world coordinate system


In [ ]:
w = science_im.wcs
print(w)


# Convert the segmentation image into a catalogue

In [ ]:
source_table=SourceCatalog(science_im.data, segimage, error=None, mask=None, background=None, wcs=w)
print('Source table length:\n', len(source_table))
print('\n\n')
print('First entry centroid x and y values :') 
print(source_table[0].centroid[0], source_table[0].centroid[1])
print('\n\n')
print('First source sky icrs coordinates :') # 
print(source_table[0].sky_centroid_icrs)
print(source_table[0].sky_centroid_icrs.ra, source_table[0].sky_centroid_icrs.dec)


# Define positions for photometry 

In [ ]:
positions = []   

for star in source_table:
    positions.append((star.centroid[0], star.centroid[1]))
print(positions[0:10]) # delete 
# Comment on this
apertures = CircularAperture(positions, r=20.0)


# Measure photometry 

In [ ]:
phot_table = aperture_photometry(science_im, apertures)
print(phot_table)

# Gaia query for a star 

#### I found this star using the science image and the segmentation image - how?

In [ ]:
idx=951-1
print(phot_table[idx])
print(source_table[idx].sky_centroid_icrs.ra, source_table[idx].sky_centroid_icrs.dec)

rad=float(source_table[idx].sky_centroid_icrs.ra.value)
decd=float(source_table[idx].sky_centroid_icrs.dec.value)
print(rad,decd)
# Get coordinates in astropy format
coord = SkyCoord(ra=rad, dec=decd, unit=(u.degree, u.degree), frame='icrs')
# Match radius in degrees
radius = u.Quantity(0.001, u.deg)
# Run query and return results
j = Gaia.cone_search_async(coord, radius)
r = j.get_results()
if len(r)>0:
    print('Gaia BP mag: ', r[0]['phot_bp_mean_mag'])
    print('Gaia G  mag: ', r[0]['phot_g_mean_mag'])
    print('Gaia RP mag: ', r[0]['phot_rp_mean_mag'])
    gmag=[r[0]['phot_bp_mean_mag'], r[0]['phot_g_mean_mag'], r[0]['phot_rp_mean_mag']]
    

# Determine V-band magnitude for this star and zeropoint

#### Here we are using one star in our image to get the zeropoint. It is better in practice to use multiple stars. 

#### Blue stars are best for measuring the zeropoint as they have a small colour term

In [ ]:
# Gaia-V relation from an earlier notebook 
Vpoly = [0.4438681, 0.26560882, 0.0039465 ]

V = gmag[1] + Vpoly[0]*(gmag[0]-gmag[1])**2 + Vpoly[1]*(gmag[0]-gmag[1]) + Vpoly[2]

print('V mag: ', V)

zp = V + 2.5*np.log10(phot_table[idx]['aperture_sum'].value)
print('Zeropoint: ', zp) 

# Determine photometry for all the other objects 

\begin{equation}
m = zp - 2.5 {\rm log} ( F)
\end{equation}

In [ ]:
Vmags=[]
for phot in phot_table['aperture_sum']:
    Vmags.append(zp-2.5*np.log10(phot.value))

print('Example V-band magnitudes') 
print(Vmags[100:120])

# Model

#### We can identify pixels with values consistent with a model 

#### At each pixel we fit the model light distribution of a star to the data 

#### Peaks will occur at the positons of stars (or peaks in the noise too)

#### Individual pixels do not have to meet a signal-to-noise threshold 

####

# Model fitting

Consider fitting a model to the data (for example, a 2D Gaussain) - $f(x,y)$

We could also normalise this model so 

\begin{equation}
\int \int f(x,y) dx dy = 1
\end{equation}

At every pixel we can try to fit the model to the data, with peaks in the model corresponding to objects

\begin{equation}
\chi ^2 = \sum_i  \frac{(d(x_i,y_i) - A f(x_i,y_i))^2} {\sigma_i^2}
\end{equation}

\begin{equation}
\chi ^2 = \sum_i  \frac{d(x_i,y_i)^2 - 2 A d(x_i,y_i) f(x_i,y_i) + A^2 f(x_i,y_i)^2 } {\sigma_i^2}
\end{equation}

where $A$ is a scaling factor, $d(x,y)$ is the data (i.e. counts) and $\sigma$ is the noise. 

The best fit value of A can be found by taking the derivative of this model and finding when the derivative equals zero.

\begin{equation}
0 = \frac{d\chi ^2}{dA} = \sum_i  \frac{ - 2 d(x_i,y_i) f(x_i,y_i) + 2 A f(x_i,y_i)^2 } {\sigma_i^2}
\end{equation}

\begin{equation}
 \sum_i  \frac{ 2 d(x_i,y_i) f(x_i,y_i)} {\sigma_i^2} = \sum_i  \frac{ + 2 A f(x_i,y_i)^2 } {\sigma_i^2}
\end{equation}

\begin{equation}
 \sum_i  \frac{ d(x_i,y_i) f(x_i,y_i)} {\sigma_i^2} = A \sum_i  \frac{ f(x_i,y_i)^2 } {\sigma_i^2}
\end{equation}

\begin{equation}
A = \sum_i  \frac{ d(x_i,y_i) f(x_i,y_i)} {\sigma_i^2} \div \sum_i  \frac{ f(x_i,y_i)^2 } {\sigma_i^2}
\end{equation}

If we find the pixels where $A$ peaks then these peaks will correspond to objects (at least when they are significantly above the background).

If the noise is approximately uniform across the image then the equation can be simplified, and one is effectively just multiplying the 2D data with the 2D model (i.e. a convolution).

Also, if the model is a good approximation of the data then $A$ is equal to the object flux. 

DAOStarFinder and IRAFStarFinder in astropy both use this approach.